# Verified geoprocessing, end to end

MapSmith's core promise: **every output ships with a provenance manifest and passes deterministic verification** — CRS discipline, geometry validity, count invariants — before it reaches you.

This notebook runs a classic two-step analysis (buffer wells by 500 m, clip to a zone of interest) using the same engine functions that back MapSmith's MCP tools. Everything is synthetic and self-contained: just `pip install mapsmith` and run.

In [1]:
import json
from pathlib import Path

import geopandas as gpd
from shapely.geometry import Point, box

data = Path('data'); data.mkdir(exist_ok=True)

wells = gpd.GeoDataFrame(
    {'name': ['well_a', 'well_b', 'well_c', 'well_d']},
    geometry=[Point(9.19, 45.46), Point(9.21, 45.475),
              Point(9.17, 45.45), Point(9.23, 45.49)],
    crs='EPSG:4326',
)
wells.to_file(data / 'wells.gpkg')

zone = gpd.GeoDataFrame({'zone': ['center']},
                        geometry=[box(9.16, 45.44, 9.22, 45.48)], crs='EPSG:4326')
zone.to_parquet(data / 'zone.parquet')
wells

,name,geometry
0,well_a,POINT (9.19 45.46)
1,well_b,POINT (9.21 45.475)
2,well_c,POINT (9.17 45.45)
3,well_d,POINT (9.23 45.49)


## Step 1 — metric buffer on a geographic CRS

The wells are in EPSG:4326 (degrees). A 500 m buffer in degrees would be wrong, so the engine reprojects to an estimated UTM zone, buffers, comes back — and **records that decision** in the manifest.

In [2]:
from mapsmith.engines import vector

buffered = vector.buffer(str(data / 'wells.gpkg'), 500.0,
                         str(data / 'wells_500m.parquet'))
buffered

{'output': 'data\\wells_500m.parquet',
 'feature_count': 4,
 'provenance': 'data\\wells_500m.parquet.provenance.json',
 'verified': True}

## Step 2 — clip to the zone of interest

In [3]:
clipped = vector.clip(str(data / 'wells_500m.parquet'),
                      str(data / 'zone.parquet'),
                      str(data / 'wells_at_risk.parquet'))
clipped

{'output': 'data\\wells_at_risk.parquet',
 'feature_count': 3,
 'provenance': 'data\\wells_at_risk.parquet.provenance.json',
 'verified': True}

## The receipt: provenance + verification

Every output has a `<name>.provenance.json` next to it: inputs with sha256, exact parameters, motivated CRS decisions, engine + version, and the deterministic checks that ran. Checks are written **before** any failure is raised — the audit trail survives errors.

In [4]:
manifest = json.loads(Path(str(data / 'wells_500m.parquet') +
                          '.provenance.json').read_text())
print(json.dumps(manifest, indent=2)[:1500])

{
  "operation": "buffer_layer",
  "parameters": {
    "distance_meters": 500.0
  },
  "inputs": [
    {
      "path": "data\\wells.gpkg",
      "sha256": "d423f60990fba6d148afaf142a34ee4df2df5eb6bcb81fdbca4a0be4014ed50d",
      "crs": "EPSG:4326"
    }
  ],
  "crs_decisions": {
    "analysis_crs": "EPSG:32632",
    "reason": "estimated UTM zone for metric buffering on a geographic CRS"
  },
  "engine": {
    "name": "geopandas",
    "version": "1.1.4"
  },
  "verification": [
    {
      "name": "crs_present",
      "passed": true,
      "detail": "{\"$schema\": \"https://proj.org/schemas/v0.7/projjson.schema.json\", \"type\": \"GeographicCRS\", \"name\": \"WGS 84\", \"datum_ensemble\": {\"name\": \"World Geodetic System 1984 ensemble\", \"members\": [{\"name\": \"World Geodetic System 1984 (Transit)\"}, {\"name\": \"World Geodetic System 1984 (G730)\"}, {\"name\": \"World Geodetic System 1984 (G873)\"}, {\"name\": \"World Geodetic System 1984 (G1150)\"}, {\"name\": \"World Geodetic S

In [5]:
from mapsmith import preview

for layer in preview.map_preview([str(data / 'wells_500m.parquet'),
                                  str(data / 'wells_at_risk.parquet')])['layers']:
    p = layer['provenance']
    print(f"{layer['name']:>18}: {layer['feature_count']} features · "
          f"{p['operation']} · {p['engine']} · "
          f"{'verified ✓' if p['verified'] else 'NOT verified'}")

        wells_500m: 4 features · buffer_layer · geopandas · verified ✓
     wells_at_risk: 3 features · clip_layer · geopandas · verified ✓


In an MCP client (Claude, ChatGPT, VS Code…) the same operations are the `buffer_layer` / `clip_layer` tools, and `preview_map` renders these layers on an interactive in-chat map with the provenance cards.